# CatBoost on tweets translated to English (Google Translate) with cleaning experiments

In [ ]:
DATA_DIR = "../data"

In [ ]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv', sep=";")
train_df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,RT Anies dapat tepuk tangan meriah saat jadi R...,Politik
2,@CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...,Demografi
3,RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE [^\]]*\]', '', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")

    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [ ]:
train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
train_df = drop_duplicated(train_df)
train_df.head()

Df size: (5000, 2)
After dropping empty text rows, df size:  (4991, 2)
After dropping literal duplicated rows, df size:  (4325, 2)
After dropping text duplicated rows, df size:  (4208, 2)


,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik


In [ ]:
train_df.to_csv(f"{DATA_DIR}/train_cleaned_v2.csv", index=False)

In [ ]:
train_df.label.value_counts()

label
Politik                    2853
Sosial Budaya               381
Ekonomi                     260
Pertahanan dan Keamanan     254
Ideologi                    253
Sumber Daya Alam            133
Demografi                    55
Geografi                     19
Name: count, dtype: int64

In [ ]:
from deep_translator import GoogleTranslator

def translate_to_en(text):
    return GoogleTranslator(source='id', target='en').translate(text)

In [ ]:
translate_to_en("saya suka makan")

'I like to eat'

In [ ]:
import translators as ts

# _ = ts.preaccelerate_and_speedtest(translator="google")

def translate_to_en_v2(text):
    return ts.translate_text(text, translator="google", from_language="id", to_language='en')

def translate_to_id_v2(text):
    return ts.translate_text(text, translator="google", from_language="en", to_language='id')

In [ ]:
translate_to_en_v2("apa yg prabowo blg tentang gaza cuma tunjukin dia tuh tone deaf and it s really disgusting dia ga peduli tentang genosida samsek masa sih dia lupa kalau palestine under occupation selama 75 thn lebih military checkpoints illegal settlements dll kacau deh")

"What is Prabowo yet about Gaza just showed him tone deaf and it really disaggent he didn't care about the genocide samatek when he forgot that Palestine under Occupation for 75 years more military checkpoints illegal settlements etc."

In [ ]:
train_df["text_en"] = train_df["text"].apply(translate_to_en_v2)
train_df["text_id"] = train_df["text_en"].apply(translate_to_id_v2)
train_df.to_csv(f"{DATA_DIR}/train_cleaned_v2_translated.csv", index=False)
train_df.head()

,text,label,text_en,text_id
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,Prabowo's visit was to inaugurate and submit a...,Kunjungan Prabowo akan meresmikan dan menyerah...
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik,Anies can be applauded when becoming the Chanc...,Anies dapat bertepuk tangan saat menjadi kanse...
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi,"emng really, supporter 01 there is a stupid th...","EMNG Sungguh, pendukung 01 Ada bodoh yang juga..."
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik,When Anies was critical of Pak Prabowo's perfo...,Ketika anies kritis terhadap kinerja Pak Prabo...
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik,Anies Baswedan hopes that the State Civil Appa...,Anies Baswedan berharap bahwa peralatan sipil ...


In [ ]:
# def get_feat(df):
#     # flag ekonomi
#     df["is_ekonomi"] = df["text"].str.contains("ekonomi", case=False).astype(int)
#     df["is_umkm"] = df["text"].str.contains("umkm", case=False).astype(int)
#     df["is_utang"] = df["text"].str.contains("utang", case=False).astype(int)
#
#     # demografi
#     df["is_generasi"] = df["text"].str.contains("generasi", case=False).astype(int)
#     df["is_muda"] = df["text"].str.contains("muda", case=False).astype(int)
#     df["is_gen"] = df["text"].str.contains("gen", case=False).astype(int)
#
#     # geografi
#     df["is_pulau"] = df["text"].str.contains("pulau", case=False).astype(int)
#     df["is_daerah"] = df["text"].str.contains("daerah", case=False).astype(int)
#
#     # ideologi
#     df["is_indonesia"] = df["text"].str.contains("indonesia", case=False).astype(int)
#     df["is_nilai"] = df["text"].str.contains("nilai", case=False).astype(int)
#     df["is_dasar"] = df["text"].str.contains("dasar", case=False).astype(int)
#     df["is_pemimpin"] = df["text"].str.contains("pemimpin", case=False).astype(int)
#     df["is_prinsip"] = df["text"].str.contains("prinsip", case=False).astype(int)
#     df["is_pancasila"] = df["text"].str.contains("pancasila", case=False).astype(int)
#
#     # pertahanan dan keamanan
#     df["is_pertahanan"] = df["text"].str.contains("pertahanan", case=False).astype(int)
#     df["is_keamanan"] = df["text"].str.contains("keamanan", case=False).astype(int)
#     df["is_alutsista"] = df["text"].str.contains("alutsista", case=False).astype(int)
#     df["is_tni"] = df["text"].str.contains("tni", case=False).astype(int)
#     df["is_polisi"] = df["text"].str.contains("polisi", case=False).astype(int)
#     df["is_militer"] = df["text"].str.contains("militer", case=False).astype(int)
#     df["is_tentara"] = df["text"].str.contains("tentara", case=False).astype(int)
#     df["is_siber"] = df["text"].str.contains("siber", case=False).astype(int)
#
#     # sosial budaya
#     df["is_budaya"] = df["text"].str.contains("budaya", case=False).astype(int)
#     df["is_anak"] = df["text"].str.contains("anak", case=False).astype(int)
#     df["is_pendidikan"] = df["text"].str.contains("pendidikan", case=False).astype(int)
#     df["is_perempuan"] = df["text"].str.contains("perempuan", case=False).astype(int)
#
#     # sumber daya alam
#     df["is_alam"] = df["text"].str.contains("alam", case=False).astype(int)
#     df["is_pertanian"] = df["text"].str.contains("pertanian", case=False).astype(int)
#     df["is_lingkungan"] = df["text"].str.contains("lingkungan", case=False).astype(int)
#     df["is_air"] = df["text"].str.contains("air", case=False).astype(int)
#     df["is_sumber"] = df["text"].str.contains("sumber", case=False).astype(int)
#
#     return df

In [ ]:
train_df = get_feat(train_df)
train_df.head()

,text,label,is_ekonomi,is_umkm,is_utang,is_generasi,is_muda,is_gen,is_pulau,is_daerah,...,is_siber,is_budaya,is_anak,is_pendidikan,is_perempuan,is_alam,is_pertanian,is_lingkungan,is_air,is_sumber
0,kunjungan prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,anies dapat tepuk tangan meriah saat jadi rekt...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,emng bener sih pendukung 01 ada yg goblok begi...,Demografi,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,sewaktu anies bersikap kritis ke kinerja pak p...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,anies baswedan harap asn termasuk tni dan polr...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
train_df.columns

Index(['text', 'label', 'is_ekonomi', 'is_umkm', 'is_utang', 'is_generasi',
       'is_muda', 'is_gen', 'is_pulau', 'is_daerah', 'is_indonesia',
       'is_nilai', 'is_dasar', 'is_pemimpin', 'is_prinsip', 'is_pancasila',
       'is_pertahanan', 'is_keamanan', 'is_alutsista', 'is_tni', 'is_polisi',
       'is_militer', 'is_tentara', 'is_siber', 'is_budaya', 'is_anak',
       'is_pendidikan', 'is_perempuan', 'is_alam', 'is_pertanian',
       'is_lingkungan', 'is_air', 'is_sumber'],
      dtype='object')

In [ ]:
train_df

,text,label
0,kunjungan prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,anies dapat tepuk tangan meriah saat jadi rekt...,Politik
2,emng bener sih pendukung 01 ada yg goblok begi...,Demografi
3,sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,anies baswedan harap asn termasuk tni dan polr...,Politik
...,...,...
4995,ngeliat debat kemaren pas prabowo kicep kekira...,Politik
4996,masyarakat yakin bahwa prabowogibran memiliki ...,Politik
4997,imo both are irrational but yg satu jauh lebih...,Ekonomi
4998,look at that pak ganjar anda sdh berkecimpung ...,Pertahanan dan Keamanan


In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

In [ ]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    # data=train_df_split.drop("label", axis=1).to_numpy(),
    data=train_df_split[["text_en"]].to_numpy(),
    label=train_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=['text',
                   # 'is_ekonomi', 'is_umkm', 'is_utang', 'is_generasi',
                   # 'is_muda', 'is_gen', 'is_pulau', 'is_daerah', 'is_indonesia',
                   # 'is_nilai', 'is_dasar', 'is_pemimpin', 'is_prinsip', 'is_pancasila',
                   # 'is_pertahanan', 'is_keamanan', 'is_alutsista', 'is_tni', 'is_polisi',
                   # 'is_militer', 'is_tentara', 'is_siber', 'is_budaya', 'is_anak',
                   # 'is_pendidikan', 'is_perempuan', 'is_alam', 'is_pertanian',
                   # 'is_lingkungan', 'is_air', 'is_sumber'
                   ],
)

val_pool = Pool(
    # data=val_df_split.drop("label", axis=1).to_numpy(),
    data=val_df_split[["text_en"]].to_numpy(),
    label=val_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=['text',
                   # 'is_ekonomi', 'is_umkm', 'is_utang', 'is_generasi',
                   # 'is_muda', 'is_gen', 'is_pulau', 'is_daerah', 'is_indonesia',
                   # 'is_nilai', 'is_dasar', 'is_pemimpin', 'is_prinsip', 'is_pancasila',
                   # 'is_pertahanan', 'is_keamanan', 'is_alutsista', 'is_tni', 'is_polisi',
                   # 'is_militer', 'is_tentara', 'is_siber', 'is_budaya', 'is_anak',
                   # 'is_pendidikan', 'is_perempuan', 'is_alam', 'is_pertanian',
                   # 'is_lingkungan', 'is_air', 'is_sumber'
                   ]
)

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42
)

cb_model.fit(
    train_pool,
    eval_set=val_pool
)

Learning rate set to 0.118869
0:	learn: 0.6333696	test: 0.6195688	best: 0.6195688 (0)	total: 12.2ms	remaining: 12.2s
100:	learn: 0.7089575	test: 0.6661548	best: 0.6693212 (82)	total: 1.29s	remaining: 11.5s
200:	learn: 0.7336150	test: 0.6733746	best: 0.6770664 (187)	total: 2.52s	remaining: 10s
300:	learn: 0.7499965	test: 0.6762283	best: 0.6823960 (291)	total: 3.73s	remaining: 8.67s
400:	learn: 0.7690417	test: 0.6741408	best: 0.6823960 (291)	total: 4.93s	remaining: 7.36s
500:	learn: 0.7863336	test: 0.6732309	best: 0.6823960 (291)	total: 6.13s	remaining: 6.11s
600:	learn: 0.8021024	test: 0.6794026	best: 0.6826482 (544)	total: 7.32s	remaining: 4.86s
700:	learn: 0.8175960	test: 0.6789847	best: 0.6826482 (544)	total: 8.52s	remaining: 3.63s
800:	learn: 0.8282554	test: 0.6806914	best: 0.6826482 (544)	total: 9.67s	remaining: 2.4s
900:	learn: 0.8418726	test: 0.6787350	best: 0.6826482 (544)	total: 10.6s	remaining: 1.17s
999:	learn: 0.8540032	test: 0.6794777	best: 0.6826482 (544)	total: 11.6s	rema

In [ ]:
# Make predictions on the validation set
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.30015804589311634


In [ ]:
val_df_split.shape

(848, 2)

In [ ]:
len(predictions)

848

In [ ]:
val_df_split

,text,label
0,sejak oktober sampai sekarang serangan ke prab...,Politik
1,laksamana madya tni purn agus setiadji eks sek...,Pertahanan dan Keamanan
2,keputusan pak ganjar pak mahfud untuk menginte...,Pertahanan dan Keamanan
3,supir di samarinda mengeluhkan pembangunan jal...,Politik
4,utamakan gagasan dan diskusi anies kampanye jo...,Politik
...,...,...
843,prabowo jangan termakan cuci otak indonesia ne...,Ideologi
844,gue suka deh sama keberagaman keunggulan pasan...,Politik
845,prabowo sosok yang menggetarkan jawa barat den...,Politik
846,prabowo meningkatkan nasionalisme itu omong ko...,Politik


In [ ]:
val_df_split.head(10)

,text,label
0,gua beneran sedih banget kenapa ya orang tuh s...,Politik
1,Siti Atikoh mengunjungi kantor TPD Ganjar Mahf...,Politik
2,Bukannya pak prabowo nantangin bahas datanya t...,Politik
3,Bahaya ini capres boneka masa data pertahanan ...,Politik
4,Elektabilitas Anies Naik di Survei Australia R...,Politik
5,Wilfrida Soik salah satu TKW yang bebas dari a...,Politik
6,Kemampuan komunikasi publik pak Anies Baswedan...,Politik
7,Mas Fadli Mas Fadli biar gak cuma omon omon do...,Sumber Daya Alam
8,gw benci bgt kalo MISALNYA lu ngomong gini ter...,Politik
9,Politics aside this is how adults should treat...,Sosial Budaya


In [ ]:
predictions

array([['Politik'],
       ['Politik'],
       ['Pertahanan dan Keamanan'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Pertahanan dan Keamanan'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Pertahanan dan Keamanan'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Sosial Budaya'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Pertahanan dan Keamanan'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['Ekonomi'],
       ['Politik'],
       ['Politik'],
       ['Ekonomi'],
       ['Politik'],
       ['Politik'],
       ['Politik'],
       ['I

In [ ]:
val_df_split["label"]

0                      Politik
1                      Politik
2                      Politik
3                      Politik
4                      Politik
                ...           
837              Sosial Budaya
838                    Politik
839                    Politik
840    Pertahanan dan Keamanan
841              Sosial Budaya
Name: label, Length: 842, dtype: object

In [ ]:
val_df_split['pred'] = predictions
val_df_split[['text', 'label', 'pred']]

ValueError: 2

In [ ]:
model_1 = CatBoostClassifier(
    text_features=['text'],
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=0,
    random_state=42
)


cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42)
cv_results_model_1 = cross_val_score(estimator=model_1,  X=train_df[["text"]], y=train_df["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[CV] START .....................................................................
[CV] END ................................ score: (test=0.330) total time=  11.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   11.0s remaining:    0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.329) total time=   9.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:   20.1s remaining:    0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.368) total time=   9.3s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:   29.5s remaining:    0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.367) total time=   9.2s
[Parallel(n_jobs=1)]: Done   4 